In [ ]:
import numpy as np
from openai import OpenAI

client = OpenAI(api_key="your-api-key")

# 1. Our "Database" (The Knowledge Base)
documents = [
    "The capital of France is Paris.",
    "The pyramids of Giza were built for the pharaoh Khufu.",
    "Python is a high-level programming language created by Guido van Rossum.",
    "The James Webb Space Telescope was launched in December 2021."
]

def get_embedding(text):
    """Converts text into a numerical vector."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# 2. Pre-compute embeddings for our documents
doc_embeddings = [get_embedding(doc) for doc in documents]

def cosine_similarity(a, b):
    """Calculates how close two vectors are."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query, top_k=1):
    """Finds the most relevant document for a query."""
    query_vec = get_embedding(query)
    
    # Compare query to every doc in our list
    similarities = [cosine_similarity(query_vec, doc_vec) for doc_vec in doc_embeddings]
    
    # Get the index of the best match
    best_idx = np.argmax(similarities)
    return documents[best_idx]

# 3. The RAG Flow
def generate_answer(query):
    # Step A: Retrieve relevant context
    context = retrieve(query)
    
    # Step B: Augment the prompt
    prompt = f"""
    Use the following context to answer the question.
    Context: {context}
    Question: {query}
    Answer:"""
    
    # Step C: Generate
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Test it out!
query = "Who made the Python language?"
print(generate_answer(query))